# 03 — Modeling

Loads , trains three classifiers, and compares results.
Run  first to generate the input file.

## 1. Imports & configuration

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, roc_curve, auc
)

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT      = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR  = Path(os.environ.get("DATA_DIR", ROOT / "data"))
DATA_PATH = DATA_DIR / "processed_data.csv"

assert DATA_PATH.exists(), f"Run 02_preprocessing.ipynb first. Not found: {DATA_PATH}"
print(f"Loading from: {DATA_PATH}")

## 2. Load processed data & reconstruct splits

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded: {df.shape}")
print(df["_split"].value_counts())

cat_cols = df.select_dtypes(include=["object", "category", "str"]).columns.tolist()
cat_cols = [c for c in cat_cols if c != "_split"]

def get_split(split_name):
    subset = df[df["_split"] == split_name].drop(columns=["_split"] + cat_cols)
    X = subset.drop(columns=["Investment_Choice"])
    # Coerce any straggler non-numeric values
    X = X.apply(pd.to_numeric, errors="coerce").fillna(X.median())
    y = subset["Investment_Choice"]
    return X, y

X_train, y_train = get_split("train")
X_test,  y_test  = get_split("test")
X_val,   y_val   = get_split("val")

print(f"
Features: {X_train.shape[1]}")
print(f"Train: {len(X_train)} | Test: {len(X_test)} | Val: {len(X_val)}")

## 3. Scaling

Fit scaler on training data only; transform test/val separately.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
X_val_scaled   = scaler.transform(X_val)

## 4. PCA

Component count is chosen to explain ≥ 95% of variance rather than being hardcoded.

In [ ]:
# Fit PCA on full component set first to find the elbow
pca_full = PCA(random_state=42)
pca_full.fit(X_train_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_components_95 = int(np.searchsorted(cumvar, 0.95)) + 1
print(f"Components needed for 95% explained variance: {n_components_95}")

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(cumvar) + 1), cumvar, marker="o", markersize=3)
plt.axhline(0.95, color="red", linestyle="--", label="95% threshold")
plt.axvline(n_components_95, color="orange", linestyle="--", label=f"{n_components_95} components")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA: choosing component count")
plt.legend()
plt.tight_layout()
plt.show()

# Apply PCA with data-driven component count
pca = PCA(n_components=n_components_95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)
X_val_pca   = pca.transform(X_val_scaled)
print(f"PCA output shape (train): {X_train_pca.shape}")

## 5. Evaluation helper

Single function used for all three models — avoids variable-reuse bugs.

In [ ]:
def evaluate_model(name, y_true, y_pred, y_prob=None):
    """Print and return a metrics dict for one model/split combination."""
    metrics = {
        "model":     name,
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "roc_auc":   roc_auc_score(y_true, y_prob[:, 1]) if y_prob is not None else None,
    }
    print(f"
{'='*40}")
    print(f"Model: {name}")
    for k, v in metrics.items():
        if k != "model" and v is not None:
            print(f"  {k:<12}: {v:.4f}")
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    return metrics

## 6. K-Nearest Neighbours

KNN uses the PCA-reduced space to keep search tractable on 80k+ rows.

In [ ]:
# Cross-validate to select k
n_neighbors_values = [23, 25, 27, 29, 31]
cv_results = []
for k in n_neighbors_values:
    scores = cross_val_score(KNeighborsClassifier(n_neighbors=k), X_train_pca, y_train, cv=5, n_jobs=-1)
    cv_results.append({"k": k, "mean": scores.mean(), "std": scores.std()})
    print(f"k={k}: mean CV acc = {scores.mean():.4f} ± {scores.std():.4f}")

best_k = max(cv_results, key=lambda r: r["mean"])["k"]
print(f"
Selected k={best_k}")

# Train final model
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train_pca, y_train)

y_pred_knn_train = knn.predict(X_train_pca)
y_pred_knn_test  = knn.predict(X_test_pca)
y_pred_knn_val   = knn.predict(X_val_pca)

y_prob_knn_train = knn.predict_proba(X_train_pca)
y_prob_knn_test  = knn.predict_proba(X_test_pca)
y_prob_knn_val   = knn.predict_proba(X_val_pca)

metrics_knn_test = evaluate_model("KNN (test)",  y_test, y_pred_knn_test, y_prob_knn_test)
metrics_knn_val  = evaluate_model("KNN (val)",   y_val,  y_pred_knn_val,  y_prob_knn_val)

## 7. Random Forest

Trained on scaled (non-PCA) features — RF is invariant to scale but we keep consistency
with the preprocessed splits.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_features=5,
    max_depth=4,
    min_samples_split=100,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train_scaled, y_train)

y_pred_rf_train = rf.predict(X_train_scaled)
y_pred_rf_test  = rf.predict(X_test_scaled)
y_pred_rf_val   = rf.predict(X_val_scaled)

y_prob_rf_train = rf.predict_proba(X_train_scaled)
y_prob_rf_test  = rf.predict_proba(X_test_scaled)
y_prob_rf_val   = rf.predict_proba(X_val_scaled)

metrics_rf_test = evaluate_model("Random Forest (test)", y_test, y_pred_rf_test, y_prob_rf_test)
metrics_rf_val  = evaluate_model("Random Forest (val)",  y_val,  y_pred_rf_val,  y_prob_rf_val)

# Feature importance
fi = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(10, 6))
fi.plot(kind="barh", ax=ax, color="steelblue")
ax.invert_yaxis()
ax.set_title("Random Forest — top 20 feature importances")
plt.tight_layout()
plt.show()

## 8. Logistic Regression

Trained on PCA-reduced features (same space as KNN for a fair comparison).

In [ ]:
log_reg = LogisticRegression(max_iter=5000, solver="saga", random_state=42)
log_reg.fit(X_train_pca, y_train)

y_pred_lr_train = log_reg.predict(X_train_pca)
y_pred_lr_test  = log_reg.predict(X_test_pca)
y_pred_lr_val   = log_reg.predict(X_val_pca)

y_prob_lr_train = log_reg.predict_proba(X_train_pca)
y_prob_lr_test  = log_reg.predict_proba(X_test_pca)
y_prob_lr_val   = log_reg.predict_proba(X_val_pca)

metrics_lr_test = evaluate_model("Logistic Regression (test)", y_test, y_pred_lr_test, y_prob_lr_test)
metrics_lr_val  = evaluate_model("Logistic Regression (val)",  y_val,  y_pred_lr_val,  y_prob_lr_val)

## 9. Model comparison table

In [ ]:
all_metrics = [metrics_knn_test, metrics_knn_val, metrics_rf_test, metrics_rf_val, metrics_lr_test, metrics_lr_val]
comparison_df = pd.DataFrame(all_metrics).set_index("model")
print(comparison_df.to_string())
comparison_df

In [ ]:
# Accuracy bar chart across train/test/val for each model
models       = ["KNN", "Random Forest", "Logistic Regression"]
acc_train    = [accuracy_score(y_train, y_pred_knn_train),
                accuracy_score(y_train, y_pred_rf_train),
                accuracy_score(y_train, y_pred_lr_train)]
acc_test     = [accuracy_score(y_test, y_pred_knn_test),
                accuracy_score(y_test, y_pred_rf_test),
                accuracy_score(y_test, y_pred_lr_test)]
acc_val      = [accuracy_score(y_val, y_pred_knn_val),
                accuracy_score(y_val, y_pred_rf_val),
                accuracy_score(y_val, y_pred_lr_val)]

x = np.arange(len(models))
w = 0.25
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - w, acc_train, w, label="Train",      color="#4C9BE8", alpha=0.85)
ax.bar(x,     acc_test,  w, label="Test",       color="#E87C4C", alpha=0.85)
ax.bar(x + w, acc_val,   w, label="Validation", color="#5DB87A", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel("Accuracy")
ax.set_title("Model comparison: train / test / validation accuracy")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 10. ROC curves

In [ ]:
def plot_roc_curve(y_true, y_prob, model_name, ax):
    """Plot one ROC curve on a given axes object."""
    fpr, tpr, _ = roc_curve(y_true, y_prob[:, 1])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2, label=f"{model_name} (AUC = {roc_auc:.3f})")
    ax.plot([0, 1], [0, 1], color="gray", linestyle="--", lw=1)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"ROC — {model_name}")
    ax.legend(loc="lower right")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plot_roc_curve(y_test, y_prob_knn_test, "KNN",                 axes[0])
plot_roc_curve(y_test, y_prob_rf_test,  "Random Forest",       axes[1])
plot_roc_curve(y_test, y_prob_lr_test,  "Logistic Regression", axes[2])
plt.suptitle("ROC curves — test set", fontsize=13)
plt.tight_layout()
plt.show()

## 11. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, y_pred) in zip(axes, [
    ("KNN",                 y_pred_knn_test),
    ("Random Forest",       y_pred_rf_test),
    ("Logistic Regression", y_pred_lr_test),
]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No invest", "Invest"],
                yticklabels=["No invest", "Invest"])
    ax.set_title(name)
    ax.set_ylabel("True label")
    ax.set_xlabel("Predicted label")
plt.suptitle("Confusion matrices — test set", fontsize=13)
plt.tight_layout()
plt.show()